# 秘書 Live2D 化 Phase 1: See-through で PSD パーツ分け

**上から順に ▶ を押すだけ**で、秘書のイラストを Live2D 用の PSD(前髪/後ろ髪/眉/白目/瞳/まつ毛/口/顔/首/服…)に分解します。

- 対応環境: **Kaggle**(推奨・GPU T4 x2・週30時間無料)/ **Google Colab**(無料 T4)
- 所要時間の目安: インストール 10〜15分 + 分解 10〜20分(初回はモデル約15GBのダウンロード込み)
- 必要なもの: `secretary_bust.png`(秘書ハブの `live2d_work/input/` にあります)
- このノートブックは**パスワード等を一切使いません**(公開モデルのみ使用)

### 事前設定
- **Kaggle**: 右パネル Settings → Accelerator を **GPU T4 x2**、Internet を **ON**(電話番号認証済みアカウントが必要)。
  右パネル「Add Input → Upload」で `secretary_bust.png` をアップロード(Dataset名は何でも可)。
- **Colab**: メニュー「ランタイム → ランタイムのタイプを変更 → T4 GPU」。画像はセル3で選択します。

## 1. GPU 確認

In [ ]:
import subprocess, sys
print(sys.version)
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(out or "GPU が見つかりません")
if not out:
    raise SystemExit("GPU が有効になっていません。Kaggle: Settings → Accelerator を『GPU T4 x2』に / Colab: ランタイムのタイプを『T4 GPU』に")
cap = float(out.splitlines()[0].split(",")[-1])
if cap < 7.5:
    # See-through が使う PyTorch 2.8 (CUDA 12.8 版) は P100 など古いGPUに対応していない
    raise SystemExit(f"このGPU({out.splitlines()[0]})は非対応です。Kaggle は Accelerator を『GPU T4 x2』に変えてください(P100 は不可)")
print("GPU OK")

## 2. See-through のインストール(10〜15分)

In [ ]:
import os, subprocess, urllib.error, urllib.request
# Kaggle かどうかは環境変数で判定する(Colab にも /kaggle フォルダがあるため、フォルダの有無では判定できない)
ROOT = "/kaggle/working" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "/content"
REPO = ROOT + "/see-through"
COMMIT = "df019de5129d6c4b406587a14c3501669441a783"

# インターネット接続の確認(Kaggle は Settings → Internet を ON にする必要あり)
try:
    urllib.request.urlopen("https://github.com", timeout=15)
except urllib.error.HTTPError:
    pass  # 応答が返ってくれば接続はできている
except Exception as e:
    raise SystemExit("インターネットに繋がっていません。Kaggle の右パネル Settings → Internet を ON にしてから"
                     "(電話番号認証が必要)、このセルをもう一度実行してください。詳細: " + repr(e))

def sh(cmd, cwd=REPO):
    print("$", cmd)
    rc = subprocess.run(cmd, shell=True, cwd=cwd).returncode
    if rc != 0:
        raise SystemExit("失敗しました: " + cmd + "  ← この上のエラー表示を秘書に貼ってください")

if not os.path.exists(REPO + "/.git"):
    sh("git clone https://github.com/shitagaki-lab/see-through.git " + REPO, cwd=ROOT)
sh("git checkout -q " + COMMIT)
# PyTorch: Kaggle/Colab に最初から入っている CUDA 版(2.8以上)があればそれを使う。
# 入れ替えると環境の CUDA ライブラリと食い違い、CUBLAS_STATUS_NOT_INITIALIZED で落ちることがあるため
pre = subprocess.run(["python", "-c", "import torch;print(torch.__version__, torch.version.cuda)"],
                     capture_output=True, text=True).stdout.split()
ver = tuple(int(x) for x in pre[0].split("+")[0].split(".")[:2]) if pre else (0, 0)
if ver >= (2, 8) and pre[1] != "None":
    print("PyTorch はプリインストール版を使います:", " ".join(pre))
else:
    sh("pip install -q torch==2.8.0+cu128 torchvision==0.23.0+cu128 torchaudio==2.8.0+cu128 --index-url https://download.pytorch.org/whl/cu128")
# 推論に不要な開発用・UI用ライブラリは入れない
# (ipykernel 等を入れ替えると Colab/Kaggle 本体が不安定になるため)
SKIP = ("ipykernel", "pytest", "pyqt6", "qtpy", "lpips", "convnext_perceptual_loss")
with open(REPO + "/requirements.txt") as f:
    reqs = [l for l in f if not l.strip().lower().startswith(SKIP)]
with open(REPO + "/requirements-notebook.txt", "w") as f:
    f.writelines(reqs)
sh("pip install -q -r requirements-notebook.txt")
sh("pip install -q -r requirements-inference-bnb.txt")  # 8GB 量子化ルート(予備)用
sh("ln -sfn common/assets assets")
os.chdir(REPO)
print("インストール完了。依存関係の ERROR/警告(dependency conflicts)が出ていても問題ありません。"
      "Colab で『セッションを再起動』と出たら、再起動してからセル3へ進んでください。")

## 3. 秘書の画像を用意

In [ ]:
import glob, os, shutil
REPO = ("/kaggle/working" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "/content") + "/see-through"
if not os.path.exists(REPO + "/inference"):
    raise SystemExit("See-through がまだインストールされていません。セル2を実行し、最後に『インストール完了』と出るのを確認してください。")
os.chdir(REPO)
os.makedirs("input", exist_ok=True)
found = glob.glob("/kaggle/input/**/secretary_bust.png", recursive=True)
if found:
    shutil.copy(found[0], "input/secretary_bust.png")
else:
    try:
        from google.colab import files
        up = files.upload()  # secretary_bust.png を選択
        shutil.move(next(iter(up)), "input/secretary_bust.png")
    except ImportError:
        raise SystemExit("Kaggle: 右パネルの Add Input から secretary_bust.png をアップロードしてください")
from PIL import Image
Image.open("input/secretary_bust.png").resize((322, 640))

## 4. パーツ分け実行(10〜20分)

T4(VRAM 15GB)向けに `--group_offload`(約10GB)で実行。T4 は省メモリ版の注意計算(bf16)に対応していないため、`run_lowmem.py` で注意計算を小分けにして実行します(結果は同じ)。失敗したら自動で 8GB 量子化ルートに切り替えます。
両方失敗した場合は、それぞれのエラーの末尾をまとめて表示します。
`--tblr_split` で左右の目・眉を別レイヤーにします(片目ウインク用)。

In [ ]:
import os
os.chdir(("/kaggle/working" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "/content") + "/see-through")
open('run_lowmem.py', 'w', encoding='utf-8').write('"""See-through の推論スクリプトを、T4(VRAM 15GB)でも動くよう省メモリで実行するラッパー。\n\n1. 注意計算(attention)の分割\n   T4 などの古い GPU は bf16 の省メモリ attention に対応していないため、\n   PyTorch は attention の行列をまるごと確保する計算方式に切り替わる(1280px で約20GB)。\n   torch.nn.functional.scaled_dot_product_attention を差し替え、\n   大きな計算だけクエリ方向に分割して実行する。結果は分割しない場合と同じ。\n2. VAE デコードを1レイヤーずつ実行\n   全レイヤー(約23枚)を一度に画像へ戻すと 7GB 超を要求するため、\n   diffusers 標準の slicing(1枚ずつ変換)を有効にする。結果は同じ。\n3. メモリの断片化対策(PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True)\n\n使い方:\n  python run_lowmem.py inference/scripts/inference_psd.py --srcp ... (元のスクリプトの引数)\n"""\nimport os\nimport runpy\nimport sys\n\n# CUDA を初期化する前に設定する必要がある\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\nimport torch\nimport torch.nn.functional as F\n\n# 1回の attention で確保してよい行列サイズの目安(バイト)\nCHUNK_BYTES = int(float(os.environ.get("ATTN_CHUNK_GB", "1")) * 1024 ** 3)\n\n_orig_sdpa = F.scaled_dot_product_attention\n\n\ndef chunked_sdpa(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, **kwargs):\n    L, S = query.shape[-2], key.shape[-2]\n    batch = query.numel() // (L * query.shape[-1])  # バッチ×ヘッド数\n    row_bytes = batch * S * query.element_size()\n    if is_causal or dropout_p or kwargs or row_bytes * L <= CHUNK_BYTES:\n        return _orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p,\n                          is_causal=is_causal, scale=scale, **kwargs)\n    step = max(1, CHUNK_BYTES // row_bytes)\n    outs = []\n    for i in range(0, L, step):\n        mask = attn_mask\n        if mask is not None and mask.dim() >= 2 and mask.shape[-2] == L:\n            mask = mask[..., i:i + step, :]\n        outs.append(_orig_sdpa(query[..., i:i + step, :], key, value, attn_mask=mask, scale=scale))\n    return torch.cat(outs, dim=-2)\n\n\nF.scaled_dot_product_attention = chunked_sdpa\n\n\ndef _patch_vae_slicing():\n    from diffusers import AutoencoderKL\n\n    orig_decode = AutoencoderKL.decode\n\n    def decode(self, z, *args, **kwargs):\n        self.use_slicing = True\n        return orig_decode(self, z, *args, **kwargs)\n\n    AutoencoderKL.decode = decode\n\n\n_patch_vae_slicing()\n\n\nif __name__ == "__main__":\n    script = sys.argv[1]\n    sys.argv = sys.argv[1:]\n    sys.path.insert(0, os.path.dirname(os.path.abspath(script)))\n    runpy.run_path(script, run_name="__main__")\n')
print('run_lowmem.py', 'を作成しました')

In [ ]:
import os
os.chdir(("/kaggle/working" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "/content") + "/see-through")
import os, subprocess
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")  # T4 x2 でも1枚目だけを使う

# まず GPU 計算が動くかを確認(ここで落ちる場合はGPUの種類かドライバの問題)
test = subprocess.run(["python", "-c", "import torch; a = torch.randn(64, 64, device='cuda', dtype=torch.bfloat16); "
                       "print(torch.cuda.get_device_name(0), float((a @ a).sum()))"], env=env, capture_output=True, text=True)
print(test.stdout or test.stderr[-2000:])
if test.returncode != 0:
    raise SystemExit("GPU 計算の確認に失敗しました。上の表示とセル1の表示を秘書に貼ってください")

def run(script, extra=(), log="run.log"):
    cmd = ["python", "run_lowmem.py", f"inference/scripts/{script}", "--srcp", "input/secretary_bust.png",
           "--save_to_psd", "--tblr_split", "--save_dir", "workspace/layerdiff_output", *extra]
    print("$", " ".join(cmd))
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end=""); f.write(line)
    return p.wait()

def tail(log, n=25):
    return "".join(open(log).readlines()[-n:])

rc = run("inference_psd.py", ["--group_offload"], log="run_main.log")
if rc != 0:
    print("\n→ 標準ルートが失敗したので、量子化ルートで再実行します\n")
    rc = run("inference_psd_quantized.py", log="run_quantized.log")
if rc != 0:
    print("=" * 60)
    print("【標準ルートのエラー(末尾)】\n" + tail("run_main.log"))
    print("【量子化ルートのエラー(末尾)】\n" + tail("run_quantized.log"))
    print("=" * 60)
    raise SystemExit("失敗しました。==== で囲まれた部分を秘書に貼ってください")
!ls -la workspace/layerdiff_output

## 5. 自動検収(レイヤー一覧画像 + 検収レポート)

In [ ]:
import os
os.chdir(("/kaggle/working" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "/content") + "/see-through")
open('psd_report.py', 'w', encoding='utf-8').write('"""See-through が出力した PSD を自己検収し、レイヤー一覧画像と検収レポートを作る。\n\n使い方:\n  python live2d_work/tools/psd_report.py <PSD> [--ref 入力画像] [--out 出力フォルダ]\n\n出力:\n  <out>/layers_overview.png   レイヤー一覧(下→上の順、市松模様=透明)\n  <out>/qa_report.md          品質チェックリスト(自動判定 + 社長の目視欄)\n\n依存: psd-tools, numpy, pillow, scipy\n"""\nimport argparse\nimport math\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\nfrom psd_tools import PSDImage\nfrom scipy import ndimage\n\n# Live2D で独立していてほしいパーツ(キー: 表示名, 値: レイヤー名に含まれる語)\nREQUIRED_PARTS = {\n    "前髪": ("hairf", "front hair"),\n    "後ろ髪": ("hairb", "back hair"),\n    "顔(輪郭・肌)": ("face",),\n    "眉": ("brow",),\n    "白目": ("eyewhite", "eyebg"),\n    "瞳": ("irid",),\n    "まつ毛・上まぶた": ("lash",),\n    "口": ("mouth",),\n    "首": ("neck",),\n    "胴体(服)": ("topwear",),\n}\n# 隠れていた部分の補完(インペイント)を確認するパーツ\nINPAINT_TARGETS = {\n    "顔(前髪の下の額)": ("face",),\n    "首(襟・髪の下)": ("neck",),\n    "後ろ髪(顔・体の裏)": ("hairb", "back hair"),\n}\nFRAGMENT_RATIO = 0.002   # レイヤー面積に対してこれ未満の孤立片を「欠け・ゴミ」とみなす\nMAX_FRAGMENTS = 5\nMIN_OCCLUDED_RATIO = 0.03  # 隠れ部分のうち、これ以上が描かれていれば補完ありとみなす\nALPHA_ON = 16\n\n\ndef find(layers, keys):\n    return [l for l in layers if any(k in l["name"].lower() for k in keys)]\n\n\ndef load_layers(psd_path):\n    psd = PSDImage.open(psd_path)\n    W, H = psd.width, psd.height\n    layers = []\n    for layer in psd.descendants():  # 下(奥)→上(手前)の順\n        if layer.is_group():\n            continue\n        img = layer.topil()\n        if img is None:\n            continue\n        canvas = Image.new("RGBA", (W, H))\n        canvas.paste(img.convert("RGBA"), (layer.left, layer.top))\n        layers.append({"name": layer.name, "rgba": np.asarray(canvas), "visible": layer.is_visible()})\n    return (W, H), layers\n\n\ndef composite(layers, size):\n    out = Image.new("RGBA", size)\n    for l in layers:\n        if l["visible"]:\n            out = Image.alpha_composite(out, Image.fromarray(l["rgba"]))\n    return out\n\n\ndef fragment_count(alpha):\n    mask = alpha > ALPHA_ON\n    area = mask.sum()\n    if area == 0:\n        return 0, 0\n    lab, n = ndimage.label(mask, structure=np.ones((3, 3)))\n    sizes = ndimage.sum(mask, lab, range(1, n + 1))\n    return int((sizes < area * FRAGMENT_RATIO).sum()), int(n)\n\n\ndef occluded_painted_ratio(layers, idx):\n    """layers[idx] のうち、上のレイヤーに完全に隠れている領域の面積比。\n\n    0 なら「見えている所しか描かれていない」=補完なし。\n    """\n    mask = layers[idx]["rgba"][..., 3] > ALPHA_ON\n    if not mask.any():\n        return 0.0\n    above = np.zeros(mask.shape, dtype=np.float32)\n    for l in layers[idx + 1:]:\n        a = l["rgba"][..., 3].astype(np.float32) / 255\n        above = above + a * (1 - above)\n    return float((mask & (above > 0.95)).sum() / mask.sum())\n\n\ndef psnr(a, b, mask):\n    d = (a.astype(np.float32) - b.astype(np.float32))[mask]\n    mse = float((d ** 2).mean()) if d.size else 0.0\n    return float("inf") if mse == 0 else 10 * math.log10(255 ** 2 / mse)\n\n\ndef checker(size, s=16):\n    w, h = size\n    yy, xx = np.mgrid[0:h, 0:w]\n    c = (((xx // s) + (yy // s)) % 2 * 40 + 200).astype(np.uint8)\n    return Image.fromarray(np.stack([c, c, c, np.full_like(c, 255)], -1))\n\n\ndef font(size):\n    try:\n        return ImageFont.load_default(size=size)\n    except TypeError:  # Pillow < 10.1\n        return ImageFont.load_default()\n\n\ndef overview(layers, comp, ref, size, out_png, thumb=220):\n    W, H = size\n    tw, th = thumb, round(thumb * H / W)\n    tiles = []\n    for i, l in enumerate(layers):\n        tile = checker((W, H))\n        tile.alpha_composite(Image.fromarray(l["rgba"]))\n        tiles.append((f"{i:02d} {l[\'name\']}", tile))\n    full = checker((W, H))\n    full.alpha_composite(comp)\n    tiles.append(("== composite", full))\n    if ref is not None:\n        tiles.append(("== input", ref.convert("RGBA").resize((W, H))))\n    cols = 6\n    rows = math.ceil(len(tiles) / cols)\n    pad, label_h = 8, 22\n    sheet = Image.new("RGB", (cols * (tw + pad) + pad, rows * (th + label_h + pad) + pad), "white")\n    draw = ImageDraw.Draw(sheet)\n    f = font(15)\n    for i, (name, t) in enumerate(tiles):\n        x = pad + (i % cols) * (tw + pad)\n        y = pad + (i // cols) * (th + label_h + pad)\n        sheet.paste(t.resize((tw, th), Image.LANCZOS).convert("RGB"), (x, y + label_h))\n        draw.text((x, y + 2), name, fill="black", font=f)\n    sheet.save(out_png)\n\n\ndef mark(ok):\n    return "✅" if ok else "⚠️"\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("psd")\n    ap.add_argument("--ref", help="See-through に入れた元画像(再構成一致度の確認用)")\n    ap.add_argument("--out", help="出力フォルダ(既定: PSDと同じ場所)")\n    args = ap.parse_args()\n\n    psd_path = Path(args.psd)\n    out = Path(args.out) if args.out else psd_path.parent\n    out.mkdir(parents=True, exist_ok=True)\n    size, layers = load_layers(psd_path)\n    comp = composite(layers, size)\n    ref = Image.open(args.ref) if args.ref else None\n    overview(layers, comp, ref, size, out / "layers_overview.png")\n\n    lines = [f"# PSD 検収レポート: `{psd_path.name}`", "",\n             f"- キャンバス: {size[0]}×{size[1]} px / レイヤー数: {len(layers)}",\n             "- レイヤー一覧画像: `layers_overview.png`(番号が小さいほど奥)", ""]\n\n    # 1. 縁の欠け\n    lines += ["## 1. 各パーツの縁に欠けがない", "",\n              "| # | レイヤー | 面積(px) | 孤立片 | 判定 |", "|---|---|---|---|---|"]\n    edge_ok = True\n    for i, l in enumerate(layers):\n        frag, _ = fragment_count(l["rgba"][..., 3])\n        area = int((l["rgba"][..., 3] > ALPHA_ON).sum())\n        ok = area > 0 and frag <= MAX_FRAGMENTS\n        edge_ok &= ok\n        lines.append(f"| {i:02d} | {l[\'name\']} | {area:,} | {frag} | {mark(ok)} |")\n    lines += ["", "孤立片 = レイヤー面積の0.2%未満の切れ端。多いと縁が欠けている/他パーツの破片が混入している可能性。", ""]\n\n    # 2. インペイント\n    lines += ["## 2. 遮蔽部分がインペイントで補完されている", "",\n              "| 対象 | レイヤー | 隠れた部分まで描かれている割合 | 判定 |", "|---|---|---|---|"]\n    inpaint_ok = True\n    for label, keys in INPAINT_TARGETS.items():\n        hits = find(layers, keys)\n        if not hits:\n            inpaint_ok = False\n            lines.append(f"| {label} | (なし) | - | ⚠️ |")\n            continue\n        for l in hits:\n            r = occluded_painted_ratio(layers, layers.index(l))\n            ok = r >= MIN_OCCLUDED_RATIO\n            inpaint_ok &= ok\n            lines.append(f"| {label} | {l[\'name\']} | {r:.1%} | {mark(ok)} |")\n    lines += ["", "0%に近い = 見えている所しか無い(前髪を揺らすと穴が開く)。", ""]\n\n    # 3. 目・口の独立\n    lines += ["## 3. 目・口などのパーツが独立している", "", "| パーツ | 該当レイヤー | 判定 |", "|---|---|---|"]\n    parts_ok = True\n    for label, keys in REQUIRED_PARTS.items():\n        hits = find(layers, keys)\n        parts_ok &= bool(hits)\n        lines.append(f"| {label} | {\', \'.join(l[\'name\'] for l in hits) or \'(なし)\'} | {mark(bool(hits))} |")\n    # See-through は --tblr_split 時に "eyewhite-l" / "eyewhite-r" のように左右を分ける\n    lr = any(re.search(r"(eye|irides|eyelash)\\S*-[lr]$|^eye[lr]$", l["name"].lower()) for l in layers)\n    lines += ["", f"- 左右の目が別レイヤー(片目ウインク用): {mark(lr)}", ""]\n\n    # 4. 再構成\n    if ref is not None:\n        r = np.asarray(ref.convert("RGB").resize(size, Image.LANCZOS))\n        c = np.asarray(comp)\n        m = c[..., 3] > 200\n        p = psnr(c[..., :3], r, m)\n        lines += ["## 4. 全レイヤー合成 ≒ 元画像", "", f"- PSNR(キャラ領域): {p:.1f} dB {mark(p >= 25)}(25dB以上なら見た目はほぼ同一)", ""]\n\n    lines += ["## まとめ(自動判定)", "",\n              f"- [{\'x\' if edge_ok else \' \'}] 各パーツの縁に欠けがない",\n              f"- [{\'x\' if inpaint_ok else \' \'}] 遮蔽されていた部分(襟の下の首、前髪の下の額)がインペイントで補完されている",\n              f"- [{\'x\' if parts_ok else \' \'}] 目・口のパーツが独立している(差し替え可能)", "",\n              "## 社長の目視確認(Phase 1 承認)", "",\n              "PSD を開き(Photoshop / CLIP STUDIO / 無料なら Photopea)、各レイヤーの目アイコンを切り替えて確認:", "",\n              "- [ ] 前髪を非表示にしても、額が自然に描かれている",\n              "- [ ] 目のレイヤーを消すと、まぶたの下が肌色で埋まっている",\n              "- [ ] 口・目・眉がそれぞれ単独で動かせそうに分かれている",\n              "- [ ] 全体を表示すると元のイラストと同じに見える", "",\n              "OKなら秘書に「PSD OK」と伝えてください(=Phase 1 承認)。"]\n    (out / "qa_report.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    print(f"wrote {out / \'layers_overview.png\'} and {out / \'qa_report.md\'}")\n\n\nif __name__ == "__main__":\n    main()\n')
print('psd_report.py', 'を作成しました')

In [ ]:
import os
os.chdir(("/kaggle/working" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "/content") + "/see-through")
import glob, os, shutil, subprocess
from IPython.display import Image as IPImage, Markdown, display
psds = sorted(p for p in glob.glob("workspace/layerdiff_output/**/*.psd", recursive=True) if not p.endswith("_depth.psd"))
if not psds:
    print("workspace/layerdiff_output の中身:")
    subprocess.run("ls -laR workspace/layerdiff_output | head -60", shell=True)
    if os.path.exists("run_main.log"):
        print("\n===== セル4のログ末尾 =====\n" + "".join(open("run_main.log").readlines()[-30:]))
    raise SystemExit("PSD ができていません。セル4が最後まで終わっていない可能性があります。上の表示を秘書に貼ってください")
psd = psds[0]
os.makedirs("secretary_psd", exist_ok=True)
shutil.copy(psd, "secretary_psd/secretary_bust.psd")
depth = psd[:-4] + "_depth.psd"
if os.path.exists(depth):
    shutil.copy(depth, "secretary_psd/secretary_bust_depth.psd")
print("PSD:", psd)
r = subprocess.run(["python", "psd_report.py", "secretary_psd/secretary_bust.psd", "--ref", "input/secretary_bust.png",
                    "--out", "secretary_psd"], capture_output=True, text=True)
print(r.stdout + r.stderr)
if r.returncode != 0:
    raise SystemExit("検収スクリプトが失敗しました。上の表示を秘書に貼ってください(PSD 自体はできています。セル6でダウンロードできます)")
display(IPImage("secretary_psd/layers_overview.png"))
display(Markdown(open("secretary_psd/qa_report.md", encoding="utf-8").read()))

## 6. ダウンロード

`secretary_psd.zip` を保存し、Mac の `secretary-hub/live2d_work/secretary_psd/` に展開してください。
- Kaggle: 右パネル Output(`/kaggle/working`)から `secretary_psd.zip` をダウンロード
- Colab: 下のセルで自動ダウンロード

In [ ]:
import os
os.chdir(("/kaggle/working" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "/content") + "/see-through")
!cd secretary_psd && zip -q -r ../secretary_psd.zip . && cd .. && ls -la secretary_psd.zip
try:
    from google.colab import files; files.download("secretary_psd.zip")
except ImportError:
    import shutil; shutil.copy("secretary_psd.zip", "/kaggle/working/secretary_psd.zip")